In [ ]:
from experiment_modules import *
from numpy.testing import assert_array_equal
import numpy as onp

import matplotlib.pyplot as plt
import seaborn as sns

from copy import deepcopy
from rich.progress import track

In [ ]:
generator = Generator(50)

In [ ]:
def calculate_entropy_from_model(state, generator=generator):
    """
    Compute the entropy of a model on some data.
    """
    ntk_fn = get_ntk_function(state.apply_fn, None)

    ntk_matrix = ntk_fn(
                    generator.train_ds["inputs"],
                    generator.train_ds["inputs"],
                    {"params": state.params}
                )

    return compute_entropy(ntk_matrix)    

In [ ]:
widths = np.unique(onp.logspace(0, 3, 10, dtype=int))

In [ ]:
for depth in [4, 5]:
    entropies = {}
    for width in widths:
        entropies[width] = {}
        for _ in range(50):
            model = build_network(width, depth)()
            params, losses = train_model(1000, 50, 1e-3, model, generator)

            entropies[width]["params"] = params
            entropies[width]["losses"] = losses
    np.save(f"/data/stovey/entropy/entropies_{depth}.npy", entropies) 

In [ ]:
for key, value in entropies.items(): 

    plt.errorbar(
        value.keys(), 
        [np.mean(v) for v in value.values()], 
        yerr=[np.std(v) for v in value.values()],
        label=key
    )
plt.legend()
plt.xscale("log")
plt.show()

In [ ]:
np.save("entropies.npy", entropies)